# OmniVoice Project Studio — Google Colab (AI-native)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/binhminhanh1235/OmniVoice/blob/master/notebooks/OmniVoice_Project_Studio_Colab.ipynb)

Advanced notebook: Gradio Web UI + REST API + SSE jobs + MCP from one OmniVoice Studio server.

If you only want the temporary Gradio UI with the fewest setup steps, use `OmniVoice_Project_Studio_Colab_Gradio.ipynb`.


In [ ]:
# Install the current master branch.
!pip install -q --upgrade "git+https://github.com/binhminhanh1235/OmniVoice.git@master"

import torch
from omnivoice.hardware_quality import detect_hardware

print("CUDA:", torch.cuda.is_available())
hardware = detect_hardware()
print(hardware.summary())
for note in hardware.notes:
    print("-", note)
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime: Runtime → Change runtime type → T4 GPU")


## Workspace

Colab can use Google Drive directly for project persistence. Project state, voices, generated WAVs, `section-status.json`, `project-queue.json`, `jobs.json`, and history are stored under `MyDrive/OmniVoiceStudio`.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
WORKSPACE = "/content/drive/MyDrive/OmniVoiceStudio"
!mkdir -p "$WORKSPACE"
print("Workspace:", WORKSPACE)


## Optional stable hostname + private access

Set `USE_STABLE_TUNNEL = True` only after creating a remotely-managed Cloudflare Tunnel pointing your hostname to `http://localhost:8000`.

Create these Colab Secrets:

- `CLOUDFLARE_TUNNEL_TOKEN`
- `OMNIVOICE_API_TOKEN`
- `OMNIVOICE_UI_USERNAME`
- `OMNIVOICE_UI_PASSWORD`

When enabled, the same hostname exposes `/ui`, `/api/v1`, `/mcp`, and `/health`.


In [ ]:
PUBLIC_URL = "https://omnivoice.example.com"
USE_STABLE_TUNNEL = False

import os

if USE_STABLE_TUNNEL:
    from google.colab import userdata

    required_names = [
        "CLOUDFLARE_TUNNEL_TOKEN",
        "OMNIVOICE_API_TOKEN",
        "OMNIVOICE_UI_USERNAME",
        "OMNIVOICE_UI_PASSWORD",
    ]
    required = {}
    for name in required_names:
        try:
            required[name] = userdata.get(name)
        except Exception:
            required[name] = None
    missing = [name for name, value in required.items() if not value]
    if missing:
        raise RuntimeError("Missing Colab Secrets: " + ", ".join(missing))
    for name, value in required.items():
        os.environ[name] = value
    os.environ["OMNIVOICE_API_TOKEN_SCOPES"] = (
        "omnivoice:read,omnivoice:generate,omnivoice:queue,omnivoice:mcp"
    )
    os.environ["OMNIVOICE_PUBLIC_URL"] = PUBLIC_URL
    del required
    print("Stable private public URL configured:", PUBLIC_URL)


In [ ]:
if USE_STABLE_TUNNEL:
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod 700 /content/cloudflared
    !/content/cloudflared --version


## Launch

- Stable mode: unified server with Gradio `/ui`, REST, SSE, and MCP.
- Simple fallback: temporary Gradio share URL only.


In [ ]:
if USE_STABLE_TUNNEL:
    !omnivoice-studio serve \
      --model k2-fsa/OmniVoice \
      --workspace "$WORKSPACE" \
      --asr-model openai/whisper-small.en \
      --asr-device cpu \
      --host 0.0.0.0 \
      --port 8000 \
      --tunnel \
      --cloudflared /content/cloudflared \
      --public-url "$OMNIVOICE_PUBLIC_URL"
else:
    !omnivoice-project-studio \
      --model k2-fsa/OmniVoice \
      --workspace "$WORKSPACE" \
      --asr-model openai/whisper-small.en \
      --asr-device cpu \
      --share
